In [1]:
import torch
import torch.nn as nn
import torch.optim as opt
import matplotlib.pyplot as plt
import numpy as np

In [2]:
def training_with_clipping(model, train_data, targets,
                           optimizer, clip_value=1.0):
  optimizer.zero_grad()
  outputs = model(train_data)
  loss = nn.MSELoss()(outputs, targets)
  loss.backward()
  global_norms = nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_value)
  optimizer.step()

  return loss.item(), global_norms.item()

In [ ]:
# Modelo simples de regressão
model = nn.Sequential(
    nn.Linear(1, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

optimizer = torch.optim.SGD(model.parameters(), lr=0.03)

# Dados sintéticos: y = 3x + 2 + ruído
x = torch.linspace(-2, 2, 120).reshape(-1, 1)
y = 3 * x + 2 + 0.25 * torch.randn_like(x)

epochs = 100
clip_value = 1.0

loss_history = []
grad_norm_history = []

for epoch in range(epochs):
    # Introduz alguns lotes "difíceis" para gerar picos de gradiente
    if epoch in [29, 59, 79]:
        train_x = x * 4
        train_y = y * 4
    else:
        train_x = x
        train_y = y

    loss, grad_norm = training_with_clipping(
        model=model,
        train_data=train_x,
        targets=train_y,
        optimizer=optimizer,
        clip_value=clip_value
    )

    loss_history.append(loss)
    grad_norm_history.append(grad_norm)


# Gráfico usando as normas retornadas pela training_with_clipping()
plt.figure(figsize=(10, 6))

plt.plot(
    range(1, epochs + 1),
    grad_norm_history,
    label="Norma global antes do clipping"
)

plt.axhline(
    y=clip_value,
    linestyle="--",
    label=f"Limite de clipping = {clip_value}"
)

plt.xlabel("Época")
plt.ylabel("Norma global do gradiente")
plt.title("Gradient Norms usando training_with_clipping()")
plt.legend()
plt.grid(True)

output_path = "./training_with_clipping_grad_norms.png"
plt.savefig(output_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"Maior norma observada: {max(grad_norm_history):.4f}")
print(f"Época do maior pico: {grad_norm_history.index(max(grad_norm_history)) + 1}")
print(f"Gráfico salvo em: {output_path}")